## Initialize Phoenix and Load Dataset


In [ ]:
from phoenix.client import Client
from getpass import getpass

phoenix_base_url = input("Enter your Phoenix OTLP endpoint (e.g., https://your-phoenix.com): ").strip("/ ")
phoenix_api_key = getpass("Enter your Phoenix API key (hidden): ").strip()
phoenix_client = Client(base_url=phoenix_base_url, api_key=phoenix_api_key)

In [ ]:

from openinference.instrumentation.dspy import DSPyInstrumentor
from opentelemetry import trace as trace_api
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk import trace as trace_sdk
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource


resource = Resource.create({"service.name": "spam-filter-notebook"})

tracer_provider = trace_sdk.TracerProvider(resource=resource)
headers = {"authorization": f"Bearer {phoenix_api_key}"}

otlp_exporter = OTLPSpanExporter(endpoint=phoenix_base_url + "/v1/traces", headers=headers)

tracer_provider.add_span_processor(SimpleSpanProcessor(otlp_exporter))
trace_api.set_tracer_provider(tracer_provider)

# Instrument DSPy
DSPyInstrumentor().instrument()

In [ ]:
DSPyInstrumentor().uninstrument()

In [ ]:
dataset = phoenix_client.datasets.get_dataset(dataset="spam-classification", timeout=300)

In [ ]:
dataset.examples[0]

## Setup DSPY Language Models


In [ ]:
from getpass import getpass

openrouter_api_key = getpass("Enter your OpenRouter API key (hidden): ").strip()

#### 1. Import DSPY and Configure Language Models

In [ ]:
import dspy

llm_base_url = "https://openrouter.ai/api/v1"

teacher_lm = dspy.LM("openrouter/anthropic/claude-opus-4.6", api_base=llm_base_url, api_key=openrouter_api_key)
student_lm = dspy.LM("openrouter/x-ai/grok-4.1-fast", api_base=llm_base_url, api_key=openrouter_api_key) 

dspy.configure(lm=student_lm, adapter=dspy.JSONAdapter())

#### 2. Define Spam Classification Signature

In [ ]:
from typing import Literal

SpamLevel = Literal[
    "not_spam",
    "unlikely_spam",
    "suspicious",
    "likely_spam",
    "very_likely_spam",
    "definitely_spam",
]


class SpamClassification(dspy.Signature):
    """Analyze an email message and determine if it is spam.

    You are a cybersecurity email analyst specializing in spam detection.
    """

    text: str = dspy.InputField(desc="The plaintext body of the email message to analyze")
    subject: str = dspy.InputField(desc="The subject line key content of the email")
    return_path: str | None = dspy.InputField(desc="The Return-Path header address, indicating where bounces are sent")
    from_address: str | None = dspy.InputField(desc="The visible From header address shown to the recipient")
    received_spf: str | None = dspy.InputField(
        desc="The SPF verification result (e.g., Pass, Fail, SoftFail) extracted from headers"
    )
    reply_address: str | None = dspy.InputField(desc="The Reply-To address if different from the sender")
    authentication_results: str | None = dspy.InputField(
        desc="Technical authentication results (DKIM, SPF, DMARC) found in the headers"
    )
    reasons: list[str] = dspy.OutputField(
        desc="A list of specific observations or red flags justifying the classification"
    )
    classification: SpamLevel = dspy.OutputField(desc="The final determination of the spam risk level")


#### 3. Define Spam Classification Module

In [ ]:
class SpamClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classify = dspy.Predict(SpamClassification)

    def forward(
        self,
        text: str,
        subject: str,
        return_path: str | None = None,
        from_address: str | None = None,
        received_spf: str | None = None,
        reply_address: str | None = None,
        authentication_results: str | None = None,
    ) -> SpamClassification:
        return self.classify(
            text=text,
            subject=subject,
            return_path=return_path,
            from_address=from_address,
            received_spf=received_spf,
            reply_address=reply_address,
            authentication_results=authentication_results,
        )

#### 4. Prepare DSPY Dataset

In [ ]:
trainset = []

for example in dataset.examples:
    inputs = {
            "text": example["input"]["text"],
            "subject": example["input"]["subject"],
            "return_path": example["input"].get("return_path"),
            "from_address": example["input"].get("from_address"),
            "received_spf": example["input"].get("received_spf"),
            "reply_address": example["input"].get("reply_address"),
            "authentication_results": example["input"].get("authentication_results"),
        }
    outputs = {
            "reasons": [],
            "classification": "definitely_spam" if example["output"]["is_spam"] else "not_spam",
        }
    trainset.append(dspy.Example(**inputs, **outputs).with_inputs(*inputs.keys()))

#### 5. Define Metric

In [ ]:
LEVEL_MAP = {
    "not_spam": 0,
    "unlikely_spam": 1,
    "suspicious": 2,
    "likely_spam": 3,
    "very_likely_spam": 4,
    "definitely_spam": 5,
}

def spam_metric(example: dspy.Example, prediction: dspy.Example, trace = None) -> float:
    """Calculate a weighted score based on classification correctness and prediction confidence.

    Args:
        example: The ground-truth example containing the expected classification.
        prediction: The model's predicted output with classification and confidence.
        trace: Optional execution trace used during optimization.

    Returns:
        A float score between 0.0 and 1.0, combining a decision score (70%)
        and a calibration bonus (30%).
    """

    # Only "very_likely_spam" and "definitely_spam" trigger the spam decision.
    # We set the bar high to minimize false positives on the binary junk/keep decision.
    pred_is_spam: bool = LEVEL_MAP.get(prediction.classification, 3) > 3
    is_spam: bool = example.classification == "definitely_spam"

    if pred_is_spam == is_spam:
        decision_score: float = 1.0  # Correct classification
    elif pred_is_spam and not is_spam:
        decision_score = 0.0  # False Positive (CRITICAL FAILURE)
    else:
        decision_score = 0.3  # False Negative (Tolerable)

    # Reward predictions that land close to the ground-truth level, not just on
    # the correct side of the binary threshold.
    pred_level = LEVEL_MAP.get(prediction.classification, 3)
    true_level = LEVEL_MAP.get(example.classification, 0)
    calibration_bonus = 1.0 - abs(pred_level - true_level) / 5.0

    return 0.7 * decision_score + 0.3 * calibration_bonus

#### 6. Optimize Classifier with DSPY

In [ ]:
optimizer = dspy.MIPROv2(
    metric=spam_metric,
    auto="medium",              # balances search depth vs. cost (light | medium | heavy)
    prompt_model=teacher_lm,    # Claude Opus 4.6 — proposes instructions & bootstraps demos
    task_model=student_lm,      # Grok 4.1 Fast  — executes the optimized prompts at inference
    num_threads=10
)

In [ ]:
optimized_app = optimizer.compile(SpamClassifier(), trainset=trainset)

In [ ]:
optimized_app.save("optimized_spam_classifier_grok_4_1_fast.json")

In [ ]:
import dspy

# Define candidate models to benchmark
models = {
    "gemini-2.5-flash":  "openrouter/google/gemini-2.5-flash",
    "gemini-3":          "openrouter/google/gemini-3",
    "grok-4.1-fast":     "openrouter/x-ai/grok-4.1-fast",
    "kimi-k2.5":         "openrouter/moonshotai/kimi-k2.5",
}

results = {}

for name, model_id in models.items():
    student_lm = dspy.LM(model_id)

    optimizer = dspy.MIPROv2(
        metric=spam_metric,
        auto="medium",
        prompt_model=teacher_lm,
        task_model=student_lm,
    )

    optimized = optimizer.compile(SpamClassifier(), trainset=trainset)

    evaluator = dspy.Evaluate(devset=testset, metric=spam_metric, num_threads=4)
    score = evaluator(optimized)
    results[name] = score

    # Save the optimized program for later use
    optimized.save(f"spam_classifier_optimized_{name}.json")
    print(f"{name}: {score:.4f}")